In [ ]:
import pandas as pd
import yaml

In [ ]:
df = pd.read_csv(r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\mti-brain - testing prompt results(tables).csv")

df2 = pd.read_excel(r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\inferred_relationships 1.xlsx")

In [ ]:
df = df[["from_table", "from_column", "to_table", "to_column"]]
df.head()

In [ ]:

df2 = df2[df2['from_schema'] == 'lpp']
df2 = df2[["from_table", "from_column", "to_table", "to_column"]]
df2.head()


In [ ]:
df.count()

In [ ]:
df2.count()

In [ ]:
merged = pd.merge(df, df2, on=['from_table', 'from_column', 'to_table', 'to_column'], how='outer', indicator=True)
merged

In [ ]:
merged = merged.drop(columns=['_merge']).drop_duplicates()
merged

In [ ]:
output = {"relationships": merged.to_dict(orient="records")}

with open("relationships.yml", "w") as f:
    yaml.dump(output, f, default_flow_style=False, sort_keys=False)

print("Written to relationships.yml")

In [ ]:
all_tables = sorted(set(merged['from_table'].dropna()).union(set(merged['to_table'].dropna())))
all_tables

In [ ]:
import pandas as pd
import yaml

In [ ]:
df = pd.read_csv(r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\result 1(High).csv")


df.columns

In [ ]:
df.head()

## Compare result 1(High).csv vs lpp_semantic_model_with_descriptions.yml
Find relationships in the CSV that are missing from the YAML (checking both directions).

In [ ]:
import pandas as pd
import yaml

# Load CSV relationships
csv_df = pd.read_csv(r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\result 1(High).csv")
csv_rels = set()
for _, row in csv_df.iterrows():
    csv_rels.add((row["from_table"], row["from_column"], row["to_table"], row["to_column"]))

# Load YAML relationships
yaml_path = r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\lpp_semantic_model_with_descriptions.yml"
with open(yaml_path) as f:
    yml = yaml.safe_load(f)

yaml_rels = set()
for r in yml["relationships"]:
    yaml_rels.add((r["from_table"], r["from_column"], r["to_table"], r["to_column"]))

# Build bidirectional lookup for YAML (both directions)
yaml_bidir = set()
for ft, fc, tt, tc in yaml_rels:
    yaml_bidir.add((ft, fc, tt, tc))
    yaml_bidir.add((tt, tc, ft, fc))  # reverse direction

# Find CSV relationships NOT in YAML (either direction)
missing = []
for ft, fc, tt, tc in csv_rels:
    if (ft, fc, tt, tc) not in yaml_bidir:
        verdict = csv_df[(csv_df["from_table"] == ft) & (csv_df["from_column"] == fc) & 
                         (csv_df["to_table"] == tt) & (csv_df["to_column"] == tc)]["final_system_verdict"].values[0]
        missing.append({"from_table": ft, "from_column": fc, "to_table": tt, "to_column": tc, "verdict": verdict})

missing_df = pd.DataFrame(missing)
print(f"CSV relationships: {len(csv_rels)}")
print(f"YAML relationships: {len(yaml_rels)}")
print(f"Missing from YAML: {len(missing_df)}")
print()
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))

## Merge two join_key_profile JSONs
Merges `backend/join_key_profile.json` (1353 pairs, real profiling) and `output/join_key_profile.json` (1512 pairs, newer run).
When a pair exists in both, prefer the one with a non-dangerous verdict or actual cross_stats (i.e. the better-profiled result).

In [ ]:
import json
from datetime import datetime, timezone
from collections import Counter

FILE_A = r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\backend\join_key_profile.json"
FILE_B = r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\join_key_profile.json"
OUT    = r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\join_key_profile_merged.json"

with open(FILE_A) as f:
    data_a = json.load(f)
with open(FILE_B) as f:
    data_b = json.load(f)

print(f"File A ({data_a['generated_at']}): {data_a['total_pairs']} pairs — {data_a['summary']}")
print(f"File B ({data_b['generated_at']}): {data_b['total_pairs']} pairs — {data_b['summary']}")


def norm_key(p):
    """Canonical bidirectional key: sorted so A↔B == B↔A."""
    side_a = (p["from_table"], p["from_col"])
    side_b = (p["to_table"],   p["to_col"])
    return tuple(sorted([side_a, side_b]))


def quality_score(p):
    """Higher = better profiled. Prefer real cross_stats over all-null short-circuits."""
    if p.get("cross_stats") is not None:
        score = 2
        if p.get("verdict") == "safe":
            score += 2
        elif p.get("verdict") == "caution":
            score += 1
        return score
    return 0  # all-null or failed


# Build merged dict: key → best profile
merged: dict[tuple, dict] = {}

for p in data_a["profiles"]:
    k = norm_key(p)
    merged[k] = p

for p in data_b["profiles"]:
    k = norm_key(p)
    if k not in merged:
        merged[k] = p
    else:
        # Keep the better-profiled one
        if quality_score(p) > quality_score(merged[k]):
            # Preserve combined sources list
            existing_sources = merged[k].get("sources", [])
            new_sources = p.get("sources", [])
            merged[k] = p
            merged[k]["sources"] = sorted(set(existing_sources + new_sources))
        else:
            # Keep existing but merge sources
            existing_sources = merged[k].get("sources", [])
            new_sources = p.get("sources", [])
            merged[k]["sources"] = sorted(set(existing_sources + new_sources))

profiles = list(merged.values())

# Summary
summary = Counter(p.get("verdict", "caution") for p in profiles)

output = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "total_pairs": len(profiles),
    "source_files": [FILE_A, FILE_B],
    "summary": dict(summary),
    "profiles": profiles,
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, default=str)

print(f"\nMerged: {len(profiles)} unique pairs")
print(f"Summary: {dict(summary)}")
print(f"Written → {OUT}")

# Source overlap breakdown
src_combos = Counter(tuple(sorted(set(p.get("sources", [])))) for p in profiles)
print("\nSource combinations:")
for k, v in src_combos.most_common():
    print(f"  {k}: {v}")